In [0]:
# CELL 1 — Validate all Silver tables exist and have rows
tables = [
    "silver.fact_orders",
    "silver.fact_reviews",
    "silver.dim_restaurants",
    "silver.dim_customers",
    "silver.dim_menu_items",
]
print("=== SILVER LAYER VALIDATION ===\n")
for t in tables:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM restaurant_catalog.{t}").first().cnt
    print(f"  ✅ restaurant_catalog.{t:<30}  {count:>6,} rows")

In [0]:
# CELL 1 — Validate all Silver tables exist and have rows
tables = [
    "silver.fact_orders",
    "silver.fact_reviews",
    "silver.dim_restaurants",
    "silver.dim_customers",
    "silver.dim_menu_items",
]
print("=== SILVER LAYER VALIDATION ===\n")
for t in tables:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM restaurant_catalog.{t}").first().cnt
    print(f"  ✅ restaurant_catalog.{t:<30}  {count:>6,} rows")

In [0]:
tables = [
    "gold.revenue_by_restaurant_daily",
    "gold.orders_by_hour_restaurant",
    "gold.revenue_by_loyalty_tier",
    "gold.avg_rating_by_restaurant",
    "gold.top_selling_categories",
    "gold.payment_method_split",
]

print("=== GOLD LAYER VALIDATION ===\n")

for t in tables:
    count = spark.sql(f"SELECT count(*) as cnt from restaurant_catalog.{t}").first().cnt
    print(f"restaurant_catalog.{t} has {count} rows")

In [0]:
# CELL 2 — Validate DLT expectations worked (check no invalid ratings slipped through)
# All ratings should be between 1 and 5 — if any row has rating outside this,
# your @dlt.expect_or_drop("valid_rating") silently failed somehow
print("=== DATA QUALITY CHECK ===\n")
spark.sql("""
    SELECT
        MIN(rating)  AS min_rating,
        MAX(rating)  AS max_rating,
        COUNT(*)     AS total_reviews,
        SUM(CASE WHEN rating NOT BETWEEN 1 AND 5 THEN 1 ELSE 0 END) AS invalid_count
    FROM restaurant_catalog.silver.fact_reviews
""").show()
# Expected: min=1, max=5, invalid_count=0

In [0]:
# CELL 3 — Validate JOIN between fact and dimension works
# This is the Gold layer preview — fact_orders joined with dim_restaurants
# If this query returns 0 rows or NULL restaurant_name, your FK keys don't match
print("=== FACT-DIMENSION JOIN TEST ===\n")
spark.sql("""
    SELECT
        r.restaurant_name,
        r.city,
        COUNT(o.order_id)        AS total_orders,
        ROUND(SUM(o.total_amount), 2) AS total_revenue_aed
    FROM restaurant_catalog.silver.fact_orders  o
    JOIN restaurant_catalog.silver.dim_restaurants r
      ON o.restaurant_id = r.restaurant_id
    GROUP BY r.restaurant_name, r.city
    ORDER BY total_revenue_aed DESC
""").show(truncate=False)
# Expected: 5 rows, one per restaurant, with realistic revenue figures